In [1]:
%load_ext autoreload
%autoreload 2



%load_ext dotenv
%autoreload 2


In [2]:
import os

os.getenv('RUN_ON_VALIDATION_SET')

'false'

### Install SDG
```bash 
git clone https://github.com/Red-Hat-AI-Innovation-Team/sdg_hub.git
cd sdg_hub
pip install .[examples]
```
**⚠️ If you haven't already, run the document pre-processing notebook to create the seed data.**

In [3]:
# Third Party
from datasets import load_dataset
from dotenv import load_dotenv

# First Party
from sdg_hub import Flow, FlowRegistry
import os

# Load environment variables from .env file
load_dotenv()

/Users/mathale/redhat-projects/sdg_hub/test_nb/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [4]:
# Required to run the flow with async mode
import nest_asyncio

nest_asyncio.apply()  

In [5]:
def create_seed_data(run_on_validation=None, seed_data_path=None):
    """
    Create seed data from QuALITY Benchmark dataset.
    
    Args:
        run_on_validation (bool, optional): If True, use validation subset. If None, reads from env.
        seed_data_path (str, optional): Path to save seed data. If None, reads from env.
    
    Returns:
        datasets.Dataset: The processed corpus
    """
    # Use environment variables as defaults if not provided
    if run_on_validation is None:
        run_on_validation = os.getenv('RUN_ON_VALIDATION_SET', 'true').lower() == 'true'
    if seed_data_path is None:
        seed_data_path = os.getenv('SEED_DATA_PATH', 'seed_data_val.jsonl')
    
    # Load QuALITY Benchmark dataset
    print("Loading QuALITY Benchmark dataset...")
    quality_corpus = load_dataset("zitongyang/entigraph-quality-corpus", split='train').remove_columns(['entity', 'entigraph']).rename_columns({'raw': 'document', 'uid': 'document_outline'})
    
    # Define seed examples for knowledge tuning
    seed_examples = {
        "icl_document": (
          "The coastal town of Willow Creek, once renowned for its pristine beaches, now struggles with rampant pollution. Plastic debris and oil spills have devastated marine life, prompting a decline in tourism and fishing industries. Residents have organized weekly clean-up initiatives, but the scale of the problem overwhelms their efforts.",
          "Technologists at the local university have developed an AI-powered buoy system to combat this. The buoys, equipped with solar panels and filtration technology, can identify and absorb oil spills while collecting microplastics. Data from the buoys is shared publicly, raising awareness and pressuring corporations to adopt sustainable practices. Though costly, the project has sparked hope for revitalizing the ecosystem and economy."
        ),
        "icl_query_1": "How does the technological solution address the economic *and* environmental challenges highlighted in the document?",
        "icl_query_2": "What implicit values or priorities do the community's actions (clean-up initiatives) and the technologists' project reflect, and how do these align or contrast?",
        "icl_query_3": "Imagine the buoy project succeeds. What unintended consequences might arise from its impact, considering document's themes?",
        "domain": "articles/essays"
    }
    
    # Add seed examples to the corpus
    quality_corpus = quality_corpus.map(lambda x: seed_examples)
    
    if run_on_validation:
        # Validation set - use predefined document IDs for consistent evaluation
        DOC_UIDS = [
            ' Defining Decay Down by David Plotz',
            ' Fight Clubbed by David Plotz',
            ' I, Antichrist? by Jeffrey Goldberg',
            " It's Time To Keelhaul U-Haul! by Jeffrey Goldberg",
            " My Father's Estate by Ben Stein",
            '"Phone Me in Central Park" by McConnell, James V.',
            'A Coffin for Jacob by Ludwig, Edward W.',
            'A Fall of Glass by Lee, Stanley R.',
            'A Filbert Is a Nut by Raphael, Rick',
            'A Gift from Earth by Banister, Manly',
            'A Gleeb for Earth by Schafhauser, Charles',
            'A Good Year for the Roses? by David Edelstein',
            'A Pail of Air by Leiber, Fritz',
            'A Planet Named Joe by Hunter, Evan',
            "AI: what's the worst that could happen? by Harry Armstrong",
            'Accidental Death by Baily, Peter',
            'All Day September by Kuykendall, Roger',
            'Ambition by Bade, William L.',
            'And Then the Town Took Off by Wilson, Richard',
            'Atom Mystery [Young Atom Detective] by Coombs, Charles Ira',
            'Beach Scene by King, Marshall',
            'Big Ancestor by Wallace, F. L. (Floyd L.)',
            'Birds of a Feather by Silverberg, Robert',
            'Bodyguard by Gold, H. L. (Horace Leonard)'
        ]
        
        # Filter corpus to validation set
        quality_corpus = quality_corpus.filter(lambda x: x['document_outline'] in DOC_UIDS)
        print(f"Running on validation set with {len(quality_corpus)} documents")
    else:
        # Use full dataset for training
        print(f"Running on full dataset with {len(quality_corpus)} documents")
    
    # Save the seed data
    quality_corpus.to_json(seed_data_path, orient='records', lines=True)
    print(f"Saved seed data to: {seed_data_path}")
    
    return quality_corpus

In [6]:
# Create seed data using the function
quality_corpus = create_seed_data()


Loading QuALITY Benchmark dataset...
Running on full dataset with 263 documents


Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 18.88ba/s]

Saved seed data to: seed_data_val.jsonl


### Run SDG
- This will create knowledge flow from provided yaml file
- We will run this on small dataset for demo purposes
- For large scale generation, please use the python command provided in the next cell
- You can analyze the generated data to ensure the quality is similar to proivded QnA pairs

In [7]:
# Setup model configuration in flow object
def set_model_config(flow_object):
    model_provider = os.getenv('MODEL_PROVIDER', 'hosted_vllm')
    print(f"Using model provider: {model_provider}")
    # Set model provider
    if model_provider == 'hosted_vllm':    
        vllm_model = os.getenv('VLLM_MODEL', 'hosted_vllm/meta-llama/Llama-3.3-70B-Instruct')
        vllm_api_base = os.getenv('VLLM_API_BASE', 'http://localhost:8000/v1')
        vllm_api_key = os.getenv('VLLM_API_KEY', 'EMPTY')
        flow_object.set_model_config(
            model=vllm_model,
            api_base=vllm_api_base,
            api_key=vllm_api_key,
        )
    elif model_provider == 'openai':
        openai_api_key = os.getenv('OPENAI_API_KEY')
        openai_model = os.getenv('OPENAI_MODEL', 'openai/gpt-4')
        flow_object.set_model_config(
            model=openai_model,
            api_key=openai_api_key,
        )
    elif model_provider == 'ollama':
        ollama_model = os.getenv('OLLAMA_MODEL', 'ollama/gemma2')
        ollama_api_base = os.getenv('OLLAMA_API_BASE', 'http://localhost:11434')
        flow_object.set_model_config(
            model=ollama_model,
            api_base=ollama_api_base,
        )
    elif model_provider == 'maas':
        maas_model = os.getenv('MAAS_MODEL')
        maas_api_base = os.getenv('MAAS_API_BASE')
        maas_api_key = os.getenv('MAAS_API_KEY')
        flow_object.set_model_config(
            model=maas_model,
            api_base=maas_api_base,
            api_key=maas_api_key,
        )
    return flow_object 

In [8]:
os.getenv("MODEL_PROVIDER")

'hosted_vllm'

#### Discover the available generation flows

In [9]:
# Auto-discover all available flows (no setup needed!)
FlowRegistry.discover_flows()

# List available flows
flows = FlowRegistry.list_flows()
print(f"Available flows: {flows}")

# You can also search the flows by tag
qa_flows = FlowRegistry.search_flows(tag="question-generation")
print(f"QA flows: {qa_flows}")

[17:51:42] INFO     Discovered 6 flows                                                              ]8;id=766082;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/registry.py\registry.py]8;;\:]8;id=118926;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/registry.py#113\113]8;;\

┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID               ┃ Name                  ┃ Author               ┃ Tags                  ┃ Description           ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ epic-jade-656    │ Extractive Summary    │ SDG Hub Contributors │ knowledge-tuning,     │ Generate extractive   │
│                  │ Knowledge Tuning      │                      │ document-internaliza… │ summary from the      │
│                  │ Dataset Generation    │                      │ question-generation,  │ input document. Each  │
│                  │ Flow                  │                      │ knowledge-extractive… │ document is first     │
│                  │                       │                      │ qa-pairs,             │ converted into list   │
│                  │                       │                      │ extractive-summaries  │ of knowledge segments │
│                  │                       │                      │                       │ for creating          │
│                  │                       │                      │                       │ extractive summary    │
│                  │                       │                      │                       │ and then annotated    │
│                  │                       │                      │                       │ with context,         │
│                  │                       │                      │                       │ relationship and      │
│                  │                       │                      │                       │ relevance. This is    │
│                  │                       │                      │                       │ then converted into   │
│                  │                       │                      │                       │ Question-Answer       │
│                  │                       │                      │                       │ pairs.                │
│ green-clay-812   │ Structured Text       │ SDG Hub Contributors │ text-analysis,        │ Multi-step pipeline   │
│                  │ Insights Extraction   │                      │ summarization, nlp,   │ for extracting        │
│                  │ Flow                  │                      │ structured-output,    │ structured insights   │
│                  │                       │                      │ insights,             │ from text including   │
│                  │                       │                      │ sentiment-analysis,   │ summary, keywords,    │
│                  │                       │                      │ entity-extraction,    │ entities, and         │
│                  │                       │                      │ keyword-extraction    │ sentiment analysis    │
│                  │                       │                      │                       │ combined into a JSON  │
│                  │                       │                      │                       │ output                │
│ heavy-heart-77   │ Key Facts Knowledge   │ SDG Hub Contributors │ knowledge-tuning,     │ Generating list of    │
│                  │ Tuning Dataset        │                      │ document-internaliza… │ atomic facts from a   │
│                  │ Generation Flow       │                      │ question-generation,  │ document and          │
│                  │                       │                      │ qa-pairs, key-facts   │ converting each       │
│                  │                       │                      │                       │ atomic fact into a QA │
│                  │                       │                      │                       │ pair. This flow will  │
│                  │                       │                      │                       │ generate 5 QA pairs   │
│                  │                       │            

Available flows: [{'id': 'green-clay-812', 'name': 'Structured Text Insights Extraction Flow'}, {'id': 'small-rock-799', 'name': 'Advanced Document Grounded Question-Answer Generation Flow for Knowledge Tuning'}, {'id': 'mild-thunder-748', 'name': 'Detailed Summary Knowledge Tuning Dataset Generation Flow'}, {'id': 'heavy-heart-77', 'name': 'Key Facts Knowledge Tuning Dataset Generation Flow'}, {'id': 'stellar-peak-605', 'name': 'Document Based Knowledge Tuning Dataset Generation Flow'}, {'id': 'epic-jade-656', 'name': 'Extractive Summary Knowledge Tuning Dataset Generation Flow'}]
QA flows: [{'id': 'small-rock-799', 'name': 'Advanced Document Grounded Question-Answer Generation Flow for Knowledge Tuning'}, {'id': 'mild-thunder-748', 'name': 'Detailed Summary Knowledge Tuning Dataset Generation Flow'}, {'id': 'heavy-heart-77', 'name': 'Key Facts Knowledge Tuning Dataset Generation Flow'}, {'id': 'stellar-peak-605', 'name': 'Document Based Knowledge Tuning Dataset Generation Flow'}, {'i

In [10]:
# We will use below mapping of flow names to their respective summarization flows
flow_name_map = {
        'Detailed Summary Knowledge Tuning Dataset Generation Flow': 'gen_detailed_summary',
        'Key Facts Knowledge Tuning Dataset Generation Flow': 'gen_atomic_facts',
        'Extractive Summary Knowledge Tuning Dataset Generation Flow': 'gen_extractive_summary',
        'Document Based Knowledge Tuning Dataset Generation Flow': 'gen_document_based_qa',
    }

In [ ]:
# Generate data for extractive summary
flow_name = "Extractive Summary Knowledge Tuning Dataset Generation Flow"
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
flow = set_model_config(flow)
number_of_summaries = int(os.getenv('NUMBER_OF_SUMMARIES', '50'))
# Generate data for extractive summary
extractive_summary_generated_data = flow.generate(quality_corpus, runtime_params={
        flow_name_map[flow_name]: {
            'n': number_of_summaries
        },
    }, max_concurrency=2)
save_data_path = os.getenv('OUTPUT_DATA_FOLDER', '')
extractive_summary_generated_data.to_json(os.path.join(save_data_path, 'extractive_summary', 'gen.jsonl'), orient='records', lines=True)

print(f"✓ Extractive summary: {len(extractive_summary_generated_data)} records")

print(f"✓ Columns: {list(extractive_summary_generated_data.column_names)}")

In [ ]:
# Generate similar data for Detailed Summary
flow_name = "Detailed Summary Knowledge Tuning Dataset Generation Flow"
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
flow = set_model_config(flow)

# Generate data for detailed summary
detailed_summary_generated_data = flow.generate(quality_corpus, runtime_params={
        flow_name_map[flow_name]: {
            'n': number_of_summaries
        },
    }, max_concurrency=2)
save_data_path = os.getenv('OUTPUT_DATA_FOLDER', '')
detailed_summary_generated_data.to_json(os.path.join(save_data_path, 'detailed_summary', 'gen.jsonl'), orient='records', lines=True)

print(f"✓ Detailed summary: {len(detailed_summary_generated_data)} records")

print(f"✓ Columns: {list(detailed_summary_generated_data.column_names)}")

In [ ]:
# Generate similar data for key facts 
flow_name = "Key Facts Knowledge Tuning Dataset Generation Flow"
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
flow = set_model_config(flow)

# Generate data for key facts summary
key_facts_generated_data = flow.generate(quality_corpus, runtime_params={
        flow_name_map[flow_name]: {
            'n': number_of_summaries
        },
    }, max_concurrency=2)


save_data_path = os.getenv('OUTPUT_DATA_FOLDER', '')
key_facts_generated_data.to_json(os.path.join(save_data_path, 'key_facts_to_qa', 'gen.jsonl'), orient='records', lines=True)

print(f"✓ Key facts: {len(key_facts_generated_data)} records")

print(f"✓ Columns: {list(key_facts_generated_data.column_names)}")

In [12]:
# Generate data for document based QA
flow_name = "Document Based Knowledge Tuning Dataset Generation Flow"
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

flow = set_model_config(flow)

# Get runtime parameters
enable_reasoning = os.getenv('ENABLE_REASONING', 'false').lower() in ('1', 'true', 'yes')
if enable_reasoning:
    # Increase max tokens to accommodate reasoning content
    runtime_params = {'question_generation': {'max_tokens': 1024}}
else:
    runtime_params = {}


document_based_qa_generated_data = flow.generate(quality_corpus, 
                                                runtime_params=runtime_params,
                                                max_concurrency=100)

save_data_path = os.getenv('OUTPUT_DATA_FOLDER', '')
document_based_qa_generated_data.to_json(os.path.join(save_data_path, 'document_based_qa', 'gen.jsonl'), orient='records', lines=True)

print(f"✓ Document based QA: {len(document_based_qa_generated_data)} records")

print(f"✓ Columns: {list(document_based_qa_generated_data.column_names)}")



[17:53:29] INFO     Loading flow from:                                                                  ]8;id=397123;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=289804;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#172\172]8;;\
                    /Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/flows/qa_generation/document_gro            
                    unded_qa/enhanced_multi_summary_qa/doc_direct_qa/flow.yaml                                     

Using model provider: hosted_vllm


           INFO     Auto-detected 3 LLM blocks for configuration: ['answer_generation',                 ]8;id=753700;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=551371;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#864\864]8;;\
                    'eval_faithfulness', 'question_generation']                                                    

[17:53:29] INFO     Loaded LLM client for model 'hosted_vllm/openai/gpt-oss-120b'              ]8;id=636055;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py\client_manager.py]8;;\:]8;id=759881;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py#61\61]8;;\

[17:53:29] INFO     Initialized LLMChatBlock 'question_generation' with model                 ]8;id=968353;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/llm_chat_block.py\llm_chat_block.py]8;;\:]8;id=318422;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/llm_chat_block.py#265\265]8;;\
                    'hosted_vllm/openai/gpt-oss-120b'                                                              

...

[18:08:14] INFO     Block 'eval_faithfulness' completed successfully: 1345 samples, 19 columns          ]8;id=885481;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=321678;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#732\732]8;;\

╭────────────────────── Document Based Knowledge Tuning Dataset Generation Flow - Complete ───────────────────────╮
│                                        Flow Execution Summary                                                   │
│ ┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓           │
│ ┃ Block Name           ┃ Type            ┃   Duration ┃     Rows     ┃     Columns     ┃   Status   ┃           │
│ ┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩           │
│ │ duplicate_document_… │ DuplicateColum… │      0.02s │  263 → 263   │       +1        │     ✓      │           │
│ │ question_generation… │ PromptBuilderB… │      0.02s │  263 → 263   │       +1        │     ✓      │           │
│ │ question_generation  │ LLMChatBlock    │     19.56s │  263 → 263   │       +1        │     ✓      │           │
│ │ parse_question_list  │ TextParserBlock │      0.36s │ 263 → 1,520  │       +1        │     ✓      │           │
│ │ answer_generation_p… │ PromptBuilderB… │      0.28s │   1,520 →    │       +1        │     ✓      │           │
│ │                      │                 │            │    1,520     │                 │            │           │
│ │ answer_generation    │ LLMChatBlock    │    270.97s │   1,520 →    │       +1        │     ✓      │           │
│ │                      │                 │            │    1,520     │                 │            │           │
│ │ parse_response_dict  │ TextParserBlock │      1.22s │   1,520 →    │       +2        │     ✓      │           │
│ │                      │                 │            │    1,520     │                 │            │           │
│ │ eval_faithfulness    │ EvaluateFaithf… │    593.22s │   1,520 →    │       +4        │     ✓      │           │
│ │                      │                 │            │    1,345     │                 │            │           │
│ ├──────────────────────┼─────────────────┼────────────┼──────────────┼─────────────────┼────────────┤           │
│ │ TOTAL                │ 8 blocks        │    885.67s │ 1,345 final  │    19 final     │    8/8     │           │
│ └──────────────────────┴─────────────────┴────────────┴──────────────┴─────────────────┴────────────┘           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           INFO     Flow 'Document Based Knowledge Tuning Dataset Generation Flow' completed            ]8;id=465572;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=207153;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#620\620]8;;\
                    successfully: 1345 final samples, 19 final columns                                             

Creating json from Arrow format: 100%|██████████| 2/2 [00:01<00:00,  1.06ba/s]

✓ Document based QA: 1345 records
✓ Columns: ['document_outline', 'document', 'icl_document', 'icl_query_1', 'icl_query_2', 'icl_query_3', 'domain', 'base_document', 'question_generation_prompt', 'question_list', 'question', 'answer_generation_prompt', 'response_dict', 'response', 'parse_response_dict_reasoning_content', 'eval_faithfulness_prompt', 'raw_eval_faithfulness', 'faithfulness_explanation', 'faithfulness_judgment']


In [ ]:
document_based_qa_generated_data.to_pandas().head()

,document_outline,document,icl_document,icl_query_1,icl_query_2,icl_query_3,domain,base_document,question_generation_prompt,question_list,question,answer_generation_prompt,response_dict,response,parse_response_dict_reasoning_content,eval_faithfulness_prompt,raw_eval_faithfulness,faithfulness_explanation,faithfulness_judgment
0,Fight Clubbed by David Plotz,""" Fight Clubbed"", David Plotz, 1999.\n Fight C...","[The coastal town of Willow Creek, once renown...",How does the technological solution address th...,What implicit values or priorities do the comm...,Imagine the buoy project succeeds. What uninte...,articles/essays,""" Fight Clubbed"", David Plotz, 1999.\n Fight C...",[{'content': 'You are a very knowledgeable AI ...,{'content': '[QUESTION] How does the evolution...,How does the evolution of UFC rules—from “no‑h...,[{'content': 'You are a very knowledgeable AI ...,{'content': '**The rule‑changes in the UFC are...,**The rule‑changes in the UFC are a textbook c...,We need to answer based on the document. Summa...,[{'content': 'You are a very knowledgeable AI ...,{'content': '[Start of Explanation] The respon...,The response accurately reflects information t...,YES
1,Fight Clubbed by David Plotz,""" Fight Clubbed"", David Plotz, 1999.\n Fight C...","[The coastal town of Willow Creek, once renown...",How does the technological solution address th...,What implicit values or priorities do the comm...,Imagine the buoy project succeeds. What uninte...,articles/essays,""" Fight Clubbed"", David Plotz, 1999.\n Fight C...",[{'content': 'You are a very knowledgeable AI ...,{'content': '[QUESTION] How does the evolution...,In what ways does the author use the compariso...,[{'content': 'You are a very knowledgeable AI ...,{'content': '**The author turns the usual “UFC...,"**The author turns the usual “UFC = barbaric, ...",We need to answer based on the document. The q...,[{'content': 'You are a very knowledgeable AI ...,{'content': '[Start of Explanation] The respon...,The response’s claims are all reflected in the...,YES
2,It's Time To Keelhaul U-Haul! by Jeffrey Gold...,""" It's Time To Keelhaul U-Haul!"", Jeffrey Gold...","[The coastal town of Willow Creek, once renown...",How does the technological solution address th...,What implicit values or priorities do the comm...,Imagine the buoy project succeeds. What uninte...,articles/essays,""" It's Time To Keelhaul U-Haul!"", Jeffrey Gold...",[{'content': 'You are a very knowledgeable AI ...,{'content': '[QUESTION] How does U‑Haul’s “non...,How does U‑Haul’s “non‑reservation” policy ill...,[{'content': 'You are a very knowledgeable AI ...,{'content': '**The “non‑reservation” policy in...,**The “non‑reservation” policy in the U‑Haul s...,We need to answer based on the document. Summa...,[{'content': 'You are a very knowledgeable AI ...,{'content': '[Start of Explanation] The contex...,The context describes U‑Haul’s “not really a r...,YES


In [ ]:
print(document_based_qa_generated_data.to_pandas()['question'][1])

In what ways does the author use the comparison between UFC and traditional boxing to challenge common assumptions about violence and safety in combat sports?


In [ ]:
print(document_based_qa_generated_data.to_pandas()['parse_response_dict_reasoning_content'][1])


'We need to answer based on the document. The question: "In what ways does the author use the comparison between UFC and traditional boxing to challenge common assumptions about violence and safety in combat sports?" So we need to outline the ways author does this: showing UFC is safer, less brutal, less concussive, less risk of death, gloves purpose, cage vs ropes, etc. Also the author points out that boxing is considered sport but is actually more dangerous, with deaths, injuries, gloves causing more head trauma, rope injuries. He argues that UFC\'s rules (no gloves, chain-link fence) reduce certain injuries. He also mentions that critics call UFC barbaric but boxing kills more. Also shows that the perception of violence is cultural, not actual. He uses examples: no kills in UFC, fewer serious injuries, gloves shield knuckles not heads, rope hyperextension, UFC\'s ground fighting reduces head blows, etc. Also mentions the myth that UFC is solely about bloodshed, while boxing is portr

In [ ]:
print(document_based_qa_generated_data.to_pandas()['response'][1])


**The author turns the usual “UFC = barbaric, boxing = civilized” narrative on its head by showing, point‑by‑point, that the mixed‑martial‑arts format is often *safer* than the sport most Americans accept as the gold‑standard of combat.  He does this in three overlapping ways:**

| How the author compares the two sports | What the comparison reveals about common assumptions |
|----------------------------------------|--------------------------------------------------------|
| **Purpose of the gloves** – The article explains that boxing gloves are *designed to protect the striker’s knuckles*, not the head of the opponent. Without gloves a boxer would break his hands after a few punches, which forces him to throw many hard, concussive blows. | The assumption that “gloves make boxing safer” is false; in fact they encourage repeated head‑shots, increasing the risk of brain injury.  By contrast, UFC fighters wear thin, fingertip‑type gloves that give no cushioning, so they naturally avoid t

🎉 You now have all three types of document augmentations (detailed summaries, extractive summaries, and key facts) along with their corresponding QA pairs.

✅ Next steps:
   - Combine and curate these datasets to prepare your final training data.
   - For detailed guidance on post-processing, mixing, and formatting the data for model training (including conversion to messages format), please refer to [knowledge_mixing.ipynb](knowledge_mixing.ipynb).